# Chapter 03A - From embedding rows to prediction objectives

## We can fetch a word's numbers. But who teaches the numbers?

Take a breath before adding another model. What have you actually built so far?

In the PyTorch chapters, you learned the training routine: make a guess, measure how wrong
it was, work backward, and adjust the weights. In Chapter 01, you gave a network a running
memory so an earlier input could affect a later answer.

Then [02A and 02B](../02-tokenization-and-embeddings/README.md) dealt with words.
Split text into pieces, give each piece an ID, and use that ID to fetch a row of numbers.
That row is its embedding. You also checked which rows received gradients and why padding
should not count as an answer. Given a few ready-made prediction examples, you saw the rows change.

Before moving on, there's a question worth asking: **why change the word rows at all?**

### Were those earlier embeddings already trained?

| What we used | Where its values came from | What it actually showed |
|---|---|---|
| 02A's NumPy table | Random numbers from `rng.normal`; no training | The same ID fetches the same row, and adding different neighbours changes the result |
| 02B's `nn.Embedding` | Random at first, then updated with the prediction layer on six supplied pairs | Loss can teach the word rows as well as the predictor |
| 02B's GPT-2 tokenizer | An already-trained tokenizer vocabulary and merge rules | Real token IDs and decoding, not GPT-2 embedding vectors |
| This notebook's models | Fresh random embeddings and prediction layers for each model | Learning from the text-based questions we're about to build |

We have not loaded a pretrained language model or a pretrained embedding table here.
A fixed random seed just lets us start from the same random draw again.

### A name tag tells us who. Useful features help us decide what to do.

An arbitrary row can identify `contract`. But if the next layer has to learn everything
from that arbitrary identifier, it does all the work of discovering useful similarities.
In 02B, `contract` and `agreement` both helped predict `binds` and `protects`, while `invoice`
helped predict `lists` and `amount`. Those first two words have something in common **for
that task**, even though their IDs and initial rows don't deliberately express it.

Letting the rows learn gives the model a place to store reusable features: the same later
layers can read what helps predict shared answers across different words. We don't need
a clever number for its own sake. We want numbers that make the model's job easier.
Nor do we tell it that column 2 must mean "legal document"; those features emerge, if useful,
through the task. Similar predictions don't even require the entire rows to look similar.

**That is the missing link to meaning:** we don't want `contract` and `agreement` to be
two unrelated name tags forever. If the task teaches them shared features, the model can
reuse a pattern learned with one when handling the other in a similar setting. With enough
varied examples, learned embeddings can capture useful semantic similarities and relationships.
That is a much better reason to learn the numbers than merely watching some weights change.

But don't quite say we **equate** the words. They keep separate IDs and rows; `agreement`
doesn't always mean a legal contract. Also, words with opposite meanings can occur in similar
surroundings: `increase` and `decrease` both fit before `the price`. Similar vectors can
reflect similar usage without making words interchangeable. A static row is a reusable
starting point; the surrounding sentence is still needed to work out the meaning here.

This is a choice, not a law. Frozen random rows can work, especially with a capable predictor
and a tiny dataset to memorize. Pretrained rows can already contain useful features.
Learning rows lets us adapt the representation instead of leaving all adaptation to the
predictor. To claim it helps on new text, we'd need a held-out comparison; our small runs
show how learning works, not that learned embeddings always win.

So 02B gave us a small taste of learning useful rows, using questions chosen by hand.
**Now where do we get many more of those questions from ordinary text?** Nobody wants
to write the answer key by hand.

## Let the sentence supply its own answer key

Take five words from the legal text we already know:

```text
the non-disclosure agreement protects confidential
```

Cover the middle word:

```text
the non-disclosure _________ protects confidential
```

Now you have a question and an answer. Give the model the four uncovered words and ask it
to guess `agreement`. We know the answer because we just covered it up. No extra labels needed.
Slide along the text and do it again: one document becomes a collection of practice questions.

**But hang on - why does guessing a word improve the embeddings?**

The model fetches the four input words' rows, combines their numbers, and makes a guess.
At first, a freshly initialized table is random. It has not learned what helps this task.
When the guess is wrong, the loss sends feedback backward through the prediction layers
and into the rows they used. The optimizer adjusts both. Over many examples, the rows can
become more useful inputs for this guessing game. That does not guarantee general word meaning.

```text
fetch four rows -> combine them -> guess the missing word
                                      |
                               compare with agreement
                                      |
adjust the rows <- adjust the predictor <- loss
```

Lookup still means nothing more than **fetch this row**. It does not need a predictor.
Prediction is the exercise we chose to teach the row's numbers. We connect them during
training; they are not the same operation.

You have other choices. Start with embeddings someone has already trained and keep them fixed
while your predictor learns. Or train both here, then throw away the predictor and keep the
table for another job. Here we train both so you can follow the feedback all the way back.

One naming trap: **learning together is not sharing weights.** Our input table and output
layer have separate weights. Explicitly sharing them would be called *weight tying*;
that is not what this model does.

## That gives us a task. What might the model miss?

Our first model, CBOW, averages those four rows before guessing. Simple enough. But put the
same words in a different order and the average stays the same. The model cannot notice
a difference we erased before it could use it.

That is the thread we will follow: teach word rows, notice what gets lost, then ask for a
model that can keep it. First word order, then earlier history, then a reader that takes
in a whole source and a writer that produces a separate answer.

This is the former 02C lesson, moved here with all its experiments, including the short
encoder-decoder preview. CBOW and Skip-gram teach static word rows; they are not models
that generate text one next token at a time. They give us a small place to see the learning happen.

```text
02: pieces -> IDs -> trainable rows
03A: make practice questions from text -> train rows and predictor together
03B: preserve order in a window -> expose its visibility limit
03C: carry an RNN/LSTM state -> predict an ordered continuation
03D: separate source and answer -> reopen source states with attention
04: replace recurrence with attention and position in Transformer blocks
```

Later in this notebook, a small encoder-decoder gives you a peek at the destination:
the answer becomes a sequence, but we can keep the same training loop. Don't worry about
mastering that architecture yet. 03C builds the recurrent text model; 03D slows down the
reader/writer handoff and tests it on examples the models did not train on.

**Code-only view:** [Python companion](03A-training-skeleton-and-word-objectives.py).
The notebook shows the base class inline so the mechanism remains visible. Its Python companion
uses the identical class in [language_model.py](language_model.py), which 03C and 03D also import.
Only optimization and loss alignment are common; inference remains specific to the prediction task.

### Three prediction jobs, three different answers

The same vocabulary can support different questions. `B` is the number of examples in a batch, `V` is the number of possible vocabulary items, `S` is source length, and `T` is target length.

| Objective | What the model sees | What it predicts | Word-score shape |
|---|---|---|---|
| CBOW | Four context IDs with the center removed: `[B, 4]` | One center token per row: `[B]` | `[B, V]` — one set of word scores per example |
| Skip-gram | A center ID repeated once for each neighbor: `[B × 4]` | One neighbor per center-neighbor pair: `[B × 4]` | `[B × 4, V]` — one set of word scores per pair |
| Encoder-decoder | Source IDs `[B, S]` and earlier target IDs `[B, T-1]` | The next target token at every decoder position | `[B, T-1, V]` — one set of word scores at each step |

CBOW and Skip-gram answer one question at a time. The encoder-decoder needs room for several answer positions, so its scores gain a time axis. Keep this table as a shape reference; it isn't a ranking of the models.

# 03A - One Training Skeleton, Three Language Objectives

**Guiding question:** Can we change what the model is trying to guess without rewriting the whole training loop?

> **Last time:** you could fetch a word's row and watch a ready-made training example change it.
>
> **What is missing:** a way to make those practice questions from ordinary text. Once we have that, we can try different guessing games without rebuilding the training machinery.
>
> **What you will build:** three small models that all receive `(input IDs, target IDs)` and inherit one training loop. CBOW guesses a missing word, Skip-gram guesses its neighbours, and an encoder-decoder predicts an answer sequence using the correct previous answer tokens during practice.

## Roadmap

| Complaint | Smallest response | Evidence |
|---|---|---|
| Text is not a training batch | Tokenize, count, and assign stable IDs | Printed vocabulary and encoded documents |
| Documents are not input-target pairs | Slide a fixed-width window | Visible context/target rows and shapes |
| IDs carry no numeric features | Look up trainable rows | Tensor shape trace |
| Every model seems to demand a new loop | Inherit one training loop and `training_loss` contract | CBOW uses the base class's `fit_language_model` method |
| CBOW predicts one center from many neighbours | Reverse each lesson into center-to-neighbour pairs | Skip-gram trains with the same loader and loop |
| One target word is not a generated sequence | Let an encoder hand state to a decoder | Sequence logits and greedy continuation |
| Static objectives still discard token roles | Permute one CBOW context | Identical pooled representation and logits |

**How far this example goes:** CBOW and Skip-gram use cross-entropy over the whole vocabulary rather than negative sampling. The encoder-decoder uses fixed-length, already aligned examples, full teacher forcing during training, and no attention or padding. These toy runs prove code and shape contracts; they do not establish useful production semantics, translation quality, or free-running language ability.

## 1 - Keep the text small enough to read yourself

We could download thousands of news articles. We'd get more examples, but we'd spend our time
waiting for data instead of seeing how learning works. Keep the legal phrases from 02B.
You can read every word and check every question we make from them.

The setup fixes the random seeds so a rerun on this CPU setup is easier to compare. Different
devices or PyTorch versions can still give different results.

Our tokenizer keeps lowercase letters and hyphens. That is deliberately plain: punctuation
can wait while we follow the path from a sentence to a weight update.

In [ ]:
import math
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from numpy.lib.stride_tricks import sliding_window_view
from torch.utils.data import DataLoader, TensorDataset

SEED = 17
torch.manual_seed(SEED)
np.random.seed(SEED)

LEGAL_CORPUS = [
    "the non-disclosure agreement protects confidential information",
    "the contract protects confidential records",
    "the agreement lists payment obligations",
    "a non-disclosure clause limits disclosure",
    "the contract and agreement bind the company",
]


def tokenize(text):
    return re.findall(r"[a-z-]+", text.lower())


documents = [tokenize(text) for text in LEGAL_CORPUS]
print(f"Documents: {len(documents)}")
print("First tokenized document:", documents[0])

## 2 - Give each word an address

A document is a list of words. The model needs one integer address for each retained word, so first flatten the list of documents into one stream for counting:

```text
for each document
    for each word in that document
        append the word
```

The leading expression in the equivalent comprehension comes first because it says what to collect:

```python
[word for document in documents for word in document]
```

`Counter` then records how often each word occurs. Frequency ordering makes the table deterministic and puts common teaching examples near the front; a frequent word doesn't get a more meaningful ID.

Two views of the same table serve opposite directions:

- `itos[index]` maps an integer **index to string** for inspection.
- `stoi[word]` maps a **string to index** for model input.

Keep address zero for `<unk>`, our "word not in this vocabulary" slot. `.get(word, 0)` sends a missing word there instead of raising an error. Giving unfamiliar words a slot doesn't mean the model understands them.

**Predict:** Will `contract` and `agreement` receive nearby IDs because they are related? No. IDs are addresses; only training can organize their vectors.

In [ ]:
items = []
for document in documents:
    for word in document:
        items.append(word)

counts = Counter(items)
itos = ["<unk>"] + [word for word, _ in counts.most_common()]
stoi = {word: index for index, word in enumerate(itos)}
vocabulary_size = len(itos)

id_documents = [
    [stoi.get(word, stoi["<unk>"]) for word in document]
    for document in documents
]

print(f"Tokens counted: {len(items)}")
print(f"Vocabulary size: {vocabulary_size}")
print("First 10 index-to-string entries:", itos[:10])
print("First encoded document:", id_documents[0])
assert all(itos[index] == word for word, index in stoi.items())

## 3 - Slide along the text and make practice questions

Choose two context words on each side of a target. The complete window is therefore:

```text
[left-2, left-1, TARGET, right+1, right+2]
```

Its width is `2 * context radius + 1`: four context positions plus one center target.

`sliding_window_view` exposes every consecutive window of that width without writing four nested indexing cases. For IDs `[1, 2, 3, 4, 5, 6]` and width five, it yields:

```text
[1, 2, 3, 4, 5]
[2, 3, 4, 5, 6]
```

Each document is windowed separately. That boundary matters: concatenating documents first would create fake lessons where the last words of one document predict the first words of the next.

Documents shorter than the full window cannot provide two neighbours on both sides, so they are skipped explicitly. Finally, any window containing `<unk>` is removed. That keeps this experiment focused, but a production pipeline would measure rather than silently discard unknown-heavy examples.

In [ ]:
CONTEXT_RADIUS = 2
window_width = 2 * CONTEXT_RADIUS + 1

per_document_windows = [
    sliding_window_view(np.asarray(ids), window_width)
    for ids in id_documents
    if len(ids) >= window_width
]
windows = np.concatenate(per_document_windows)
windows = windows[~(windows == stoi["<unk>"]).any(axis=1)]

print(f"Window width: {window_width}")
print(f"All windows shape: {windows.shape}")
for row in windows[:4]:
    print([itos[index] for index in row])

### Separate what the model sees from what it must predict

The center column is the answer. The remaining columns are the evidence.

`np.delete(..., axis=1)` removes the center **column from every row**; without `axis=1`, deletion would flatten or target the wrong dimension. Selecting `windows[:, CONTEXT_RADIUS]` keeps the center item from every row as the target.

PyTorch embedding layers require integer indices, so both arrays become `torch.long` tensors:

```text
complete windows: [number of lessons, 5]
context IDs:      [number of lessons, 4]
target IDs:       [number of lessons]
```

**Predict:** For a batch of eight lessons and embedding width twelve, what shape follows lookup? `[8, 4, 12]`.

In [ ]:
contexts = torch.tensor(
    np.delete(windows, CONTEXT_RADIUS, axis=1),
    dtype=torch.long,
)
targets = torch.tensor(windows[:, CONTEXT_RADIUS], dtype=torch.long)

print("Complete windows:", windows.shape)
print("Context tensor:", tuple(contexts.shape))
print("Target tensor:", tuple(targets.shape))
print(
    "First lesson:",
    [itos[index] for index in contexts[0].tolist()],
    "->",
    itos[targets[0]],
)

assert contexts.shape == (len(windows), 2 * CONTEXT_RADIUS)
assert targets.shape == (len(windows),)

## 4 - Hand over the questions with the right answers

`TensorDataset` keeps each question beside its answer. Ask for item `i` and you get both together.

`DataLoader` hands over batches of those pairs. It can shuffle the **practice questions**, but it must keep each answer with its question. It doesn't care what we're asking the model to guess. CBOW, Skip-gram, and the encoder-decoder can all use this same pattern:

```text
for input_ids, target_ids in dataloader:
    ...
```

The helper below is all the loader code this notebook needs. Our text fits in one batch; you can choose smaller batches for a bigger dataset. A fresh, seeded shuffle generator makes these runs easier to compare.

In [ ]:
def make_dataloader(input_ids, target_ids, batch_size=None):
    dataset = TensorDataset(input_ids, target_ids)
    return DataLoader(
        dataset,
        batch_size=batch_size or len(dataset),
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )


dataloader = make_dataloader(contexts, targets)
batch_contexts, batch_targets = next(iter(dataloader))

print("Batch contexts:", tuple(batch_contexts.shape))
print("Batch targets:", tuple(batch_targets.shape))
print("First batch target:", itos[batch_targets[0]])
assert len(batch_contexts) == len(batch_targets)

## 5 - Crude attempt: give every context word an equal vote

An embedding table is a trainable matrix shaped:

```text
[vocabulary size, embedding width]
```

Passing IDs does not multiply by their numeric values. It retrieves the corresponding rows. A batch shaped `[B, C]` therefore becomes `[B, C, D]`, where `C` is the number of context positions and `D` is the embedding width.

CBOW means **continuous bag of words**. "Bag" is the clue: once we average the rows into one `[B, D]` vector, there's no seat number left for any word. Swap their positions and the average stays the same.

**Predict:** If you reverse one context, will its mean vector change?

In [ ]:
EMBEDDING_WIDTH = 12
probe_embedding = nn.Embedding(vocabulary_size, EMBEDDING_WIDTH)

one_context = contexts[:1]
reversed_context = one_context.flip(dims=[1])
one_vectors = probe_embedding(one_context)
original_mean = one_vectors.mean(dim=1)
reversed_mean = probe_embedding(reversed_context).mean(dim=1)

print("IDs:", one_context.tolist())
print("After lookup:", tuple(one_vectors.shape))
print("After context mean:", tuple(original_mean.shape))
print("Maximum difference after reversal:", (original_mean - reversed_mean).abs().max().item())

assert torch.allclose(original_mean, reversed_mean)

### Where `EmbeddingBag` fits

`nn.EmbeddingBag(..., mode="mean")` combines lookup and mean pooling. For a fixed-width 2D batch, it should match `nn.Embedding(...).mean(dim=1)` when both modules share the same weight table.

The compact layer is useful in real code, especially for variable-size bags. You will keep the two operations separate in the teaching model because the intermediate `[B, C, D]` tensor makes the information loss visible.

In [ ]:
embedding_bag = nn.EmbeddingBag(
    vocabulary_size,
    EMBEDDING_WIDTH,
    mode="mean",
)
with torch.no_grad():
    embedding_bag.weight.copy_(probe_embedding.weight)

manual_means = probe_embedding(contexts[:3]).mean(dim=1)
bag_means = embedding_bag(contexts[:3])

print("Manual mean shape:", tuple(manual_means.shape))
print("EmbeddingBag shape:", tuple(bag_means.shape))
print("Maximum difference:", (manual_means - bag_means).abs().max().item())
assert torch.allclose(manual_means, bag_means)

### Objective checkpoint 1 - CBOW

Before putting this in a class, check what goes in, what comes out, and which answer we grade:

| Question | CBOW answer |
|---|---|
| Input IDs | `[B, C]`: several context positions |
| Target IDs | `[B]`: one center word per lesson |
| Vocabulary logits | `[B, V]`: one set of word scores per example |
| Which answer goes with which scores? | logits row `b` versus center ID `b` |
| What gets combined | Which context rows appeared |
| What gets lost | Context order, because changing the order doesn't change the average |

**Predict:** Does the target have a time axis? Which exact operation removes the context-position
axis? If these answers are unclear, trace `[B, C] -> [B, C, D] -> [B, D]` before reading the model
class.

## 6 - Change the guessing game, not the training routine

Each model still has two familiar jobs:

- `__init__` sets up the trainable layers.
- `forward` turns a batch of input IDs into word scores, called **logits**.

The shared loop needs one more answer: **which correct word goes with each set of scores?**
For CBOW, that's one missing word per example. For the encoder-decoder later, it's several
answer words in order. We should not need a new optimizer loop just because the answer got longer.

`TrainableLanguageModel` keeps that bookkeeping and `fit_language_model` in one place.
You still provide the data loader, loss function, and optimizer. The model chooses its layers.
Its `targets_for_loss` method picks the answers to grade; the encoder-decoder changes that
method to skip the start marker, but keeps the same training loop.

```text
input IDs
-> model's forward
-> word scores [..., vocabulary]
-> put the correct answer beside each set of scores
-> flatten the example/position axes
-> cross-entropy loss
```

Here is CBOW's part of that shared routine:

```text
context IDs [B, C]
-> fetch rows [B, C, D]
-> average them [B, D]
-> hidden features [B, D/2]
-> word scores [B, V]
```

The base class handles practice. The subclass decides how to make a guess.

In [ ]:
class TrainableLanguageModel(nn.Module):
    """Share target alignment and optimization across language objectives."""

    def targets_for_loss(self, target_ids):
        return target_ids

    def training_loss(self, input_ids, target_ids, loss_function):
        logits = self(input_ids, target_ids)
        expected_ids = self.targets_for_loss(target_ids)
        flat_logits = logits.reshape(-1, logits.shape[-1])
        flat_targets = expected_ids.reshape(-1)
        return loss_function(flat_logits, flat_targets)

    def fit_language_model(self, dataloader, loss_function, optimizer, epochs):
        history = []

        for _ in range(epochs):
            self.train()
            total_loss = 0.0

            for input_batch, target_batch in dataloader:
                optimizer.zero_grad()
                loss = self.training_loss(input_batch, target_batch, loss_function)
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * len(input_batch)

            history.append(total_loss / len(dataloader.dataset))

        return history


class CBOW(TrainableLanguageModel):
    def __init__(self, vocabulary_size, embedding_width):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_width)
        self.hidden = nn.Linear(embedding_width, embedding_width // 2)
        self.activation = nn.ReLU()
        self.output = nn.Linear(embedding_width // 2, vocabulary_size)

    def forward(self, context_ids, target_ids=None):
        token_vectors = self.embedding(context_ids)
        pooled_context = token_vectors.mean(dim=1)
        hidden_features = self.activation(self.hidden(pooled_context))
        return self.output(hidden_features)

### Trace one untrained batch before optimizing anything

Calling `model(batch)` invokes `forward` through PyTorch's module machinery. Prefer that call over `model.forward(batch)` because hooks and framework behavior live around the forward method.

The model returns `[batch, vocabulary]`. Each row is one target-word competition. Cross-entropy will compare those scores with one integer target per lesson.

In [ ]:
torch.manual_seed(SEED)
model = CBOW(vocabulary_size, EMBEDDING_WIDTH)
untrained_logits = model(batch_contexts)

print("Input contexts:", tuple(batch_contexts.shape))
print("Vocabulary logits:", tuple(untrained_logits.shape))
assert untrained_logits.shape == (len(batch_contexts), vocabulary_size)

## 7 - Keep one training loop when the model changes

Cross-entropy checks how much probability the model gives the correct word compared with the
alternatives. It takes raw scores and does the log-softmax calculation itself. Don't add a
separate softmax first: that repeats work and makes the calculation less stable.

Every practice step follows the same order:

1. `zero_grad()` clears feedback left over from the previous step.
2. `training_loss(...)` makes the guesses and lines them up with the correct answers.
3. `backward()` works out how each participating weight affected the loss.
4. `step()` lets the optimizer adjust those weights.

The loop doesn't need to know whether we averaged word rows or passed a note through a GRU.
That is the model's job. It does need to put the model in training mode, count loss fairly
across batches, and keep using the same optimizer across epochs.

Calling `model.fit_language_model(...)` uses the method inherited from the base class.
Inside it, `self` is still our actual model, so its own `forward` and `targets_for_loss`
methods do the guessing and answer selection.

Keep PyTorch's `train()` separate: it switches training mode. Despite its name, it does not
run the practice loop.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
losses = model.fit_language_model(
    dataloader,
    criterion,
    optimizer,
    epochs=180,
)

print(f"Loss: {losses[0]:.4f} -> {losses[-1]:.4f}")

### The loss fell. What does that tell us?

If the model gives every word the same chance, cross-entropy is `log(V)`, where `V` is the
vocabulary size. That gives us a useful starting line for the plot.

Falling loss means our model is fitting these practice questions better. It does not mean
it understands legal documents. There are only eight windows here; memorizing them is entirely
possible. We'll keep that distinction when we inspect the learned rows.

In [ ]:
random_guess_loss = math.log(vocabulary_size)

fig, axis = plt.subplots(figsize=(8, 4))
axis.plot(losses, color="#2a6f97", linewidth=2)
axis.axhline(
    random_guess_loss,
    color="#d97706",
    linestyle="--",
    label=f"random guess log(V) = {random_guess_loss:.2f}",
)
axis.set(title="CBOW training loss", xlabel="epoch", ylabel="cross-entropy")
axis.legend(frameon=False)
axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Random-guess baseline: {random_guess_loss:.4f}")
print(f"Final training loss: {losses[-1]:.4f}")
assert losses[-1] < losses[0]
assert losses[-1] < random_guess_loss

## 8 - Follow the feedback back to the rows we fetched

Which input rows should change after one practice question? The ones we actually used.
If an ID appears several times, its row gets feedback from each use. A row we never fetched
has no path to this batch's loss, so its embedding gradient is zero.

That is about the **input embedding table**. The output layer scores every possible word,
so it does not follow the same used-row rule.

Below, clear the old gradients and work backward once, without taking an optimizer step.
Then check the rows we fetched against the ones we left alone.

In [ ]:
model.zero_grad()
gradient_contexts = contexts[:2]
gradient_targets = targets[:2]
gradient_loss = model.training_loss(
    gradient_contexts,
    gradient_targets,
    criterion,
)
gradient_loss.backward()

gradient_norms = model.embedding.weight.grad.norm(dim=1)
used_ids = set(gradient_contexts.reshape(-1).tolist())
rows_with_gradient = {
    index for index, norm in enumerate(gradient_norms.tolist()) if norm > 0
}

print("Used rows:", [itos[index] for index in sorted(used_ids)])
print("Rows with gradients:", [itos[index] for index in sorted(rows_with_gradient)])
assert rows_with_gradient == used_ids

## 9 - The learned embedding weights are the vectors

`model.embedding.weight` is the trainable `[V, D]` table. Inspection should not extend the training graph, so:

- `detach()` returns a tensor that no longer tracks gradient history.
- `cpu()` would move GPU data to host memory; this notebook already runs on CPU, but the explicit move keeps the inspection pattern portable.
- dividing each row by its own norm creates unit vectors.

After normalization, a matrix-vector product is cosine similarity:

```text
all normalized rows @ one normalized query row
```

The query word itself would score one, so it is excluded. `<unk>` is also excluded because it received no meaningful unknown-word training in this corpus.

Nearest neighbours reveal geometry induced by this exact objective and corpus. They do not prove dictionary-like meaning.

In [ ]:
embedding_weights = model.embedding.weight.detach().cpu()
embedding_norms = embedding_weights.norm(dim=1, keepdim=True).clamp_min(1e-12)
normalized_embeddings = embedding_weights / embedding_norms


def most_similar(word, top_k=4):
    if word not in stoi:
        return f"'{word}' is not in the vocabulary"

    similarities = normalized_embeddings @ normalized_embeddings[stoi[word]]
    similarities = similarities.clone()
    similarities[stoi[word]] = -1
    similarities[stoi["<unk>"]] = -1
    neighbour_ids = similarities.topk(min(top_k, vocabulary_size - 2)).indices
    return [
        (itos[index], round(similarities[index].item(), 3))
        for index in neighbour_ids.tolist()
    ]


for word in ["contract", "agreement", "protects", "confidential"]:
    print(f"{word:>12} -> {most_similar(word)}")

### A 2D projection is a map, not the territory

The embedding rows live in twelve dimensions. To draw them, center the selected rows and use singular value decomposition to find two directions that preserve as much variance as possible.

This PCA-style view is deterministic and useful for orientation. It still discards dimensions, so apparent 2D distance is weaker evidence than cosine similarity measured in the original space. Labels can also make sparse regions look more meaningful than they are.

The most frequent words are shown because a crowded plot of every row would trade explanation for confetti.

In [ ]:
visible_count = min(16, vocabulary_size - 1)
visible_ids = torch.tensor(
    [stoi[word] for word, _ in counts.most_common(visible_count)]
)
visible_vectors = embedding_weights[visible_ids]
centered = visible_vectors - visible_vectors.mean(dim=0, keepdim=True)
_, _, right_singular_vectors = torch.linalg.svd(centered, full_matrices=False)
coordinates = centered @ right_singular_vectors[:2].T

fig, axis = plt.subplots(figsize=(10, 7))
axis.scatter(coordinates[:, 0], coordinates[:, 1], color="#2a6f97", s=55)
for point, token_id in zip(coordinates, visible_ids.tolist()):
    axis.annotate(itos[token_id], point, xytext=(5, 5), textcoords="offset points")
axis.axhline(0, color="#9ca3af", linewidth=0.7)
axis.axvline(0, color="#9ca3af", linewidth=0.7)
axis.set(
    title="Frequent embedding rows projected from 12D to 2D",
    xlabel="projection axis 1",
    ylabel="projection axis 2",
)
axis.grid(alpha=0.15)
plt.tight_layout()
plt.show()

## 10 - Turn the question around: guess neighbours from the center

CBOW asks:

```text
[left, left, right, right] -> center
```

What if we uncover the middle word and cover one neighbour instead?

```text
center -> left neighbour
center -> left neighbour
center -> right neighbour
center -> right neighbour
```

That is **Skip-gram**. We don't need new text or a new vocabulary. Each old window gives us
four new practice questions, all using the same center word but a different neighbour as the answer.
The loader still hands over `(input IDs, target IDs)`. What those IDs stand for has changed.

**Predict:** With `N` windows and four neighbours each, how many practice questions do we get? `4N`.

In [ ]:
skipgram_inputs = targets.repeat_interleave(contexts.shape[1])
skipgram_targets = contexts.reshape(-1)
skipgram_dataloader = make_dataloader(skipgram_inputs, skipgram_targets)

print("CBOW lessons:", len(contexts))
print("Skip-gram lessons:", len(skipgram_inputs))
print(
    "First Skip-gram lesson:",
    itos[skipgram_inputs[0]],
    "->",
    itos[skipgram_targets[0]],
)

assert len(skipgram_inputs) == len(contexts) * contexts.shape[1]
assert len(skipgram_inputs) == len(skipgram_targets)

### Objective checkpoint 2 - Skip-gram

The loader still returns two tensors, but their shapes and jobs have changed:

| Question | Skip-gram answer |
|---|---|
| Input IDs | `[B]`: one repeated center ID per pair |
| Target IDs | `[B]`: one neighbouring word per pair |
| Vocabulary logits | `[B, V]`: one set of word scores per pair |
| Which answer goes with which scores? | logits row `b` versus neighbour ID `b` |
| What it can use | Center-word identity |
| What it cannot use here | Which neighbour position or direction produced the target |

**Predict:** Which CBOW axis disappeared? Why can the same center ID appear with several different
targets? The shared fit loop survives because it consumes aligned logits and IDs, not because the
two objectives mean the same thing.

### A different question, a small code change

CBOW looked up several rows and averaged them. Skip-gram receives one center ID, so there is no context axis to pool. The hidden projection and vocabulary competition can keep the same roles:

```text
center IDs [B]
-> embedding rows [B, D]
-> hidden features [B, D/2]
-> vocabulary logits [B, V]
```

Keep the same base class, loader helper, loss function, and training loop. Here we score every word in the vocabulary so nothing is hidden. With much more text, Word2Vec usually saves work through negative sampling or hierarchical softmax instead of scoring every possible word.

In [ ]:
class SkipGram(TrainableLanguageModel):
    def __init__(self, vocabulary_size, embedding_width):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_width)
        self.hidden = nn.Linear(embedding_width, embedding_width // 2)
        self.activation = nn.ReLU()
        self.output = nn.Linear(embedding_width // 2, vocabulary_size)

    def forward(self, center_ids, target_ids=None):
        center_vectors = self.embedding(center_ids)
        hidden_features = self.activation(self.hidden(center_vectors))
        return self.output(hidden_features)


torch.manual_seed(SEED)
skipgram_model = SkipGram(vocabulary_size, EMBEDDING_WIDTH)
skipgram_optimizer = torch.optim.Adam(skipgram_model.parameters(), lr=0.03)
skipgram_losses = skipgram_model.fit_language_model(
    skipgram_dataloader,
    criterion,
    skipgram_optimizer,
    epochs=220,
)

with torch.inference_mode():
    probe_center = targets[:1]
    neighbour_logits = skipgram_model(probe_center)
    predicted_ids = neighbour_logits.topk(contexts.shape[1], dim=1).indices[0]

print(f"Skip-gram loss: {skipgram_losses[0]:.4f} -> {skipgram_losses[-1]:.4f}")
print("Center:", itos[probe_center.item()])
print("Top predicted neighbours:", [itos[index] for index in predicted_ids.tolist()])

assert skipgram_losses[-1] < skipgram_losses[0]
assert skipgram_losses[-1] < math.log(vocabulary_size)
assert neighbour_logits.shape == (1, vocabulary_size)

## 11 - We taught the rows. We still lost word order.

CBOW learned to guess a missing word. Skip-gram learned to guess its neighbours.
Both keep one embedding row per word ID: **static** means that fetching the same ID gives
the same row regardless of which sentence we are reading.

CBOW has another limit. It averages the input rows, so these two arrangements give it the same input:

```text
the contract protects confidential
confidential protects contract the
```

Adding the same rows in a different order leaves their average unchanged, apart from tiny
rounding differences. The model gets one combined vector and no record of who sat where.
Its word scores therefore stay the same too.

This isn't a problem we can fix by training longer. We threw away the order before the
predictor got a chance to use it. Neither of these word objectives builds a fresh,
sentence-dependent state for each token position.

In [ ]:
model.eval()
ordered_context = contexts[:1]
permuted_context = ordered_context[:, torch.tensor([3, 1, 0, 2])]

with torch.no_grad():
    ordered_vectors = model.embedding(ordered_context).mean(dim=1)
    permuted_vectors = model.embedding(permuted_context).mean(dim=1)
    ordered_logits = model(ordered_context)
    permuted_logits = model(permuted_context)

representation_difference = (ordered_vectors - permuted_vectors).abs().max().item()
logit_difference = (ordered_logits - permuted_logits).abs().max().item()

print("Ordered:", [itos[index] for index in ordered_context[0].tolist()])
print("Permuted:", [itos[index] for index in permuted_context[0].tolist()])
print(f"Maximum representation difference: {representation_difference:.8f}")
print(f"Maximum logit difference: {logit_difference:.8f}")

assert torch.allclose(ordered_vectors, permuted_vectors)
assert torch.allclose(ordered_logits, permuted_logits)

So far, each question asked for one word. What if we want several words, in order? Give one part of the model the reading job and another the writing job. We'll call the text it reads the **source** and the answer it writes the **target**. Same legal text, but a different kind of question.

## 12 - What if the answer needs more than one word?

Skip-gram turned the question around, but still gave one set of word scores per practice question. Translation, summarization, and text continuation can ask for several answer words in order. Let's try that change on our tiny example.

Reuse each five-token window without downloading another dataset:

```text
source: [token 0, token 1, token 2]
target: [<bos>, token 3, token 4, <eos>]
```

We're finishing a phrase, not translating it. That is enough to try the reader/writer handoff without changing the text, vocabulary, loader helper, or loss function.

`<bos>` tells the decoder where generation starts. `<eos>` gives it a learnable stopping target. Every sequence has the same length here, so padding would add machinery without teaching a new fact.

In [ ]:
BOS_ID = vocabulary_size
EOS_ID = vocabulary_size + 1
sequence_itos = itos + ["<bos>", "<eos>"]
sequence_vocabulary_size = len(sequence_itos)

window_ids = torch.tensor(windows, dtype=torch.long)
source_sequences = window_ids[:, :3]
continuations = window_ids[:, 3:]
sequence_targets = torch.cat(
    [
        torch.full((len(windows), 1), BOS_ID, dtype=torch.long),
        continuations,
        torch.full((len(windows), 1), EOS_ID, dtype=torch.long),
    ],
    dim=1,
)
sequence_dataloader = make_dataloader(source_sequences, sequence_targets)

print("Source shape:", tuple(source_sequences.shape))
print("Target shape:", tuple(sequence_targets.shape))
print("Source:", [sequence_itos[index] for index in source_sequences[0].tolist()])
print("Target:", [sequence_itos[index] for index in sequence_targets[0].tolist()])

assert source_sequences.shape == (len(windows), 3)
assert sequence_targets.shape == (len(windows), 4)

### Objective checkpoint 3 - encoder-decoder sequence prediction

The writer must guess the next answer word, not copy the word it just received. Put the inputs and expected answers beside each other before reading the code:

| Question | Encoder-decoder answer |
|---|---|
| Source IDs | `[B, S]`: an ordered source sequence |
| Full target IDs | `[B, T]`: `<bos>`, continuation, `<eos>` |
| Decoder inputs | `[B, T-1]`: target without the final token |
| Vocabulary logits | `[B, T-1, V]`: one set of word scores per answer position |
| Loss targets | `[B, T-1]`: target without `<bos>` |
| What it can use | Ordered source compressed into one recurrent hidden state, plus the correct earlier answer tokens |
| What it cannot use here | Direct retrieval of individual source positions; this model has no attention |

**Predict:** Which axes flatten for cross-entropy, and which axis must remain the vocabulary axis?
This is a compact preview of encoder-decoder alignment. Stage 04 will rebuild the mechanism with
cross-attention rather than treating this one-state bottleneck as the final architecture.

During practice, show the writer the correct previous target token before asking for the next one: `<bos>` -> correct `protects` -> correct `confidential`. This is called **teacher forcing**. During generation there is no answer key, so the writer must feed its own previous guess back in instead.

### The model owns sequence alignment; the loop stays boring

The encoder reads the source rows in order and writes a final note: its hidden state. The decoder starts with that note. During practice, we also give it the correct earlier answer tokens:

```text
encoder: source IDs [B, S] -> hidden [1, B, H]
decoder input: target IDs without final token [B, T-1]
decoder logits: [B, T-1, V]
loss targets: target IDs without <bos> [B, T-1]
```

Each position now predicts the **next** answer token. Supplying the correct previous token is teacher forcing. It helps us inspect the first training run, but there is a catch: when generating, the writer must live with its own guesses. We'll test that difference, not assume practice accuracy carries over.

`EncoderDecoderLM(TrainableLanguageModel)` inherits `fit_language_model` without redefining it. The call `sequence_model.fit_language_model(...)` below is where this encoder-decoder uses the common loop. Inside that loop, `self` is the encoder-decoder instance:

```text
sequence_model.fit_language_model(...)  [inherited from TrainableLanguageModel]
-> self.training_loss(...)             [inherited]
-> self(source_ids, target_ids)        [EncoderDecoderLM.forward]
-> encode(source_ids) -> decode(target_ids[:, :-1], hidden)
-> self.targets_for_loss(target_ids)   [EncoderDecoderLM override: target_ids[:, 1:]]
-> cross-entropy -> backward -> optimizer step
```

Only `targets_for_loss` changes which answers the shared loss grades. The encoder and decoder GRUs do the reading and writing; they don't need separate practice loops.

### A quick peek at the reader and writer

We changed the guessing game without replacing the training loop. Can we also ask for more
than one answer word? In this small preview, a reader takes in the source and passes its
final note to a writer. The writer uses that note to predict the answer one token at a time.

Don't worry about mastering the whole architecture here. Next we'll slow down and give words
ordered seats in 03B, then carry a running note in 03C. In 03D we'll ask whether the writer
should have to rely on that one final note.

In [ ]:
class EncoderDecoderLM(TrainableLanguageModel):
    def __init__(self, vocabulary_size, embedding_width, hidden_width):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_width)
        self.encoder = nn.GRU(embedding_width, hidden_width, batch_first=True)
        self.decoder = nn.GRU(embedding_width, hidden_width, batch_first=True)
        self.output = nn.Linear(hidden_width, vocabulary_size)

    def encode(self, source_ids):
        _, hidden = self.encoder(self.embedding(source_ids))
        return hidden

    def decode(self, decoder_input_ids, hidden):
        decoder_states, hidden = self.decoder(
            self.embedding(decoder_input_ids),
            hidden,
        )
        return self.output(decoder_states), hidden

    def forward(self, source_ids, target_ids=None):
        if target_ids is None:
            raise ValueError("Training requires target IDs for teacher forcing.")
        hidden = self.encode(source_ids)
        logits, _ = self.decode(target_ids[:, :-1], hidden)
        return logits

    def targets_for_loss(self, target_ids):
        return target_ids[:, 1:]


torch.manual_seed(SEED)
sequence_model = EncoderDecoderLM(
    sequence_vocabulary_size,
    embedding_width=EMBEDDING_WIDTH,
    hidden_width=24,
)
sequence_optimizer = torch.optim.Adam(sequence_model.parameters(), lr=0.03)
assert EncoderDecoderLM.fit_language_model is TrainableLanguageModel.fit_language_model
sequence_losses = sequence_model.fit_language_model(
    sequence_dataloader,
    criterion,
    sequence_optimizer,
    epochs=400,
)

with torch.inference_mode():
    teacher_forced_logits = sequence_model(source_sequences, sequence_targets)
    teacher_forced_predictions = teacher_forced_logits.argmax(dim=-1)
    expected_sequence_ids = sequence_targets[:, 1:]
    teacher_forced_accuracy = (
        teacher_forced_predictions == expected_sequence_ids
    ).float().mean().item()

print(f"Seq2Seq loss: {sequence_losses[0]:.4f} -> {sequence_losses[-1]:.4f}")
print("Teacher-forced token accuracy:", f"{teacher_forced_accuracy:.0%}")
print("Sequence logits:", tuple(teacher_forced_logits.shape))

assert sequence_losses[-1] < sequence_losses[0]
assert sequence_losses[-1] < math.log(sequence_vocabulary_size)
assert teacher_forced_accuracy >= 0.9

### Teacher forcing worked - can the decoder use its own answer?

Training supplied the correct previous target token. Generation cannot. A greedy decoder must feed its own highest-logit token back into the next step:

```text
<bos> -> predicted token -> predicted token -> ... -> <eos>
```

This is a small but necessary check: take away the answer key and see what happens. We choose the highest-scoring token at each step, not several candidate answers as beam search would. Repeating these eight practice windows correctly would still be memorization, not evidence that we built a translator.

Then reverse one source. CBOW's average couldn't notice that change, but a GRU reads step by step, so its final note should change. Noticing order is a start. It doesn't tell us whether the model uses that order well.

In [ ]:
def greedy_decode(model, source_ids, bos_id, eos_id, max_steps):
    model.eval()
    hidden = model.encode(source_ids)
    current_ids = torch.full(
        (len(source_ids), 1),
        bos_id,
        dtype=torch.long,
    )
    generated = []

    with torch.inference_mode():
        for _ in range(max_steps):
            logits, hidden = model.decode(current_ids, hidden)
            next_ids = logits[:, -1].argmax(dim=-1)
            generated.append(next_ids)
            current_ids = next_ids.unsqueeze(1)
            if bool((next_ids == eos_id).all()):
                break

    return torch.stack(generated, dim=1)


generated_ids = greedy_decode(
    sequence_model,
    source_sequences[:1],
    BOS_ID,
    EOS_ID,
    max_steps=sequence_targets.shape[1] - 1,
)

with torch.inference_mode():
    ordered_hidden = sequence_model.encode(source_sequences[:1])
    reversed_hidden = sequence_model.encode(source_sequences[:1].flip(dims=[1]))

expected_words = [
    sequence_itos[index]
    for index in sequence_targets[0, 1:].tolist()
]
generated_words = [
    sequence_itos[index]
    for index in generated_ids[0].tolist()
]
hidden_difference = (ordered_hidden - reversed_hidden).abs().max().item()

print("Expected continuation:", expected_words)
print("Greedy continuation:", generated_words)
print(f"Hidden-state difference after reversal: {hidden_difference:.6f}")

assert generated_ids.shape[0] == 1
assert generated_ids.shape[1] >= 1
assert hidden_difference > 0

### Static rows, pooled context, and recurrent state are different objects

| Object | What it knows | What it cannot know here |
|---|---|---|
| Static embedding row | Which vocabulary address was retrieved | The sentence, position, or current meaning |
| CBOW pooled vector | Which rows appeared in the local bag | Their order or a separate state for each position |
| Skip-gram center vector | Which center row must predict neighbours | Which neighbour position is being predicted |
| Encoder hidden state | The source tokens in recurrent order | Direct access to every earlier state once compressed |
| Transformer contextual state | Token identity plus information gathered for one position | Nothing automatically; access depends on masks, position, attention, and training |

CBOW and Skip-gram are Word2Vec objectives: they learn useful static rows, but they are not free-running language models. The encoder-decoder is a conditional sequence model, yet this tiny version sees correct previous answer tokens during training and has no attention.

## What changes when the model gets bigger?

| Teaching mechanism | Production counterpart | What changes | What does not change |
|---|---|---|---|
| `make_dataloader(input_ids, target_ids)` | Task-specific collation and distributed loading | Padding, masks, sharding, and scale | Batches still preserve input-target alignment |
| `training_loss` contract | Framework training step | Mixed precision, accumulation, metrics, and distributed reduction | Logits must align with target positions |
| CBOW full-vocabulary loss | Word2Vec CBOW with negative sampling | Efficient sampled objective and corpus scale | Context predicts center |
| Skip-gram full-vocabulary loss | Word2Vec Skip-gram with negative sampling | Efficient sampled objective and subsampling | Center predicts neighbours |
| GRU encoder-decoder | Translation or summarization model | Variable lengths, masks, attention, and beam search | Source conditions target generation |
| Recurrent bottleneck | Cross-attention over source states | Decoder retrieves rather than compressing everything once | Target tokens remain autoregressive |

### Your turn

1. Change `CONTEXT_RADIUS` from two to one. Predict how many Skip-gram pairs each window creates.
2. Replace CBOW mean pooling with sum pooling. Reordering will still be invisible; explain why.
3. Set the sequence source to two tokens instead of three and retrain. Compare the continuation accuracy without claiming the tiny corpus generalizes.

These drills change one known variable. None turns the toy corpus into a production language model.

**Forward: one learned summary note is a real access limit.**

The reader hands over only its final summary note. If a detail didn't survive in that note, the writer can't reopen an earlier source position to find it. [03D](03D-recurrent-encoder-decoder-to-attention.ipynb) will compare that handoff with a writer allowed to look back at the reader's saved notes.

## 13 - What did we actually find out?

| Opening question | Evidence produced here |
|---|---|
| How does text become lessons? | Per-document windows produced CBOW, Skip-gram, and source-target sequence tensors |
| What stayed reusable? | One `DataLoader` factory, `training_loss` contract, cross-entropy, and inherited `fit_language_model` method trained all three models |
| What changed for Skip-gram? | The center and context swapped roles; model input became one ID and targets became neighbouring IDs |
| What changed for sequence-to-sequence? | Targets gained a time axis, the encoder produced recurrent state, and the decoder predicted shifted target positions |
| Are embeddings learned weights? | Gradient inspection identified exactly the retrieved CBOW input rows |
| Did optimization beat useful baselines? | All three final losses were asserted below their initial values and random-guess baselines |
| Did the models acquire production semantics or generation quality? | Not established; every result came from a tiny training corpus |
| What breaks next? | Static objectives lose token roles; the recurrent encoder compresses the source into one bottleneck |

### What we tried, explained, and left for later

**What we built and checked:** token counting, per-document windows, one reusable loader factory, a documented model training contract, explicit CBOW, generic optimization, gradient participation, cosine neighbours, PCA projection, Skip-gram pair construction, full-softmax Skip-gram, fixed-length encoder-decoder training, greedy decoding, sequence accuracy, recurrent order sensitivity, and CBOW permutation invariance.

**What we explained:** Word2Vec objective direction, negative-sampling boundary, teacher forcing, static versus contextual representations, recurrent source bottleneck, and toy-to-production mappings.

**What we're leaving for later:** large-corpus embedding quality, subsampling frequent words, negative sampling implementation, variable-length padding and masks, scheduled sampling, attention, beam search, held-out model selection, and production translation. They belong later because implementing all of them here would hide the reusable skeleton.

**Next:** [Stage 04: Transformers](../../04-transformers/README.md) gives every position its own state. Its encoder-decoder chapter later replaces the one-vector source bottleneck with cross-attention.

### Next: give the words seats

You can now make practice questions from text and use them to train word rows and a predictor
together. You also saw a preview of a reader handing a note to a writer.

But let's not skip the problem we left behind: **averaging erased word order.**
Would keeping the words in separate seats fix everything? Continue to
[03B - Counts and Fixed Windows](03B-counts-and-fixed-window-language-models.ipynb).
The Stage 04 links above are a look ahead; 03B-03D build the bridge to get there.